# Défi 2 NLP (medRxiv 2026) — Projet 2 : AI-Written Abstract Detection
**Approche :** Machine Learning Classique (Stylométrie + TF-IDF hybride & Régression Logistique pondérée)  
**Auteur :** Redouane Kenniche

---

### Objectif du notebook
Ce notebook implémente un modèle de classification binaire (`0` = Humain, `1` = IA) capable de généraliser à des modèles génératifs inédits en combinant :
1. **Des caractéristiques stylométriques** (variabilité de la longueur des phrases, ponctuation, précision numérique, diversité lexicale) ;
2. **Une vectorisation lexicale et morphologique** (`TF-IDF` mots et n-grammes de caractères) ;
3. **Un calibrage du seuil de probabilité** pour maximiser directement la métrique officielle **F1-score** (avec la **précision** en critère de départage).

## 1. Importation des bibliothèques et chargement robuste des données
Importation des outils Scikit-learn et SciPy, puis recherche automatique des fichiers d'entraînement et de test (compressés ou non, à la racine ou dans un sous-dossier).

In [1]:
import html
import os
import re
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler


def trouver_fichier(noms_possibles):
    cibles = set(noms_possibles)
    for racine, _, fichiers in os.walk("."):
        for f in fichiers:
            if f in cibles:
                return os.path.join(racine, f)
    return None


def lire_csv_robuste(filepath):
    for enc in ["utf-8", "latin-1", "cp1252"]:
        for sep in [",", ";"]:
            try:
                df = pd.read_csv(filepath, encoding=enc, sep=sep)
                df.columns = df.columns.str.strip().str.strip('"')
                if "abstract" in {c.lower() for c in df.columns}:
                    return df
            except Exception:
                continue
    raise ValueError(f"Impossible de lire : {filepath}")


TRAIN_FILE = trouver_fichier([
    "ClassificationsHumanAI_AbstractsTrainSet.csv",
    "ClassificationsHumanAI_AbstractsTrainSet.csv.gz",
])
TEST_FILE = trouver_fichier([
    "ClassificationsHumanAI_AbstractsTestSet.csv",
    "ClassificationsHumanAI_AbstractsTestSet.csv.gz",
])

df_train = lire_csv_robuste(TRAIN_FILE)
print(f"Jeu d'entraînement chargé : {df_train.shape} | Colonnes : {list(df_train.columns)}")

Jeu d'entraînement chargé : (10000, 4) | Colonnes : ['ID', 'category', 'abstract', 'HumanAI']


## 2. Extraction d'indices stylométriques et nettoyage léger
Contrairement à la classification thématique, la détection de texte généré par IA repose fortement sur la **ponctuation**, la **casse**, les **nombres** et la **régularité des phrases** (*burstiness*). Le nettoyage retire donc uniquement les balises HTML tout en préservant la structure fine du texte.

In [2]:
def nettoyer_html_leger(texte):
    if not isinstance(texte, str):
        return ""
    texte = html.unescape(texte)
    texte = re.sub(
        r"</?(?:p|b|i|u|sub|sup|br|em|strong|span|div|ul|ol|li)\b[^>]*>",
        " ",
        texte,
    )
    return re.sub(r"\s+", " ", texte).strip()


class StylometricExtractor(BaseEstimator, TransformerMixin):
    """Extrait des marqueurs stylistiques discriminants entre humains et LLM."""

    MARQUEURS_IA = (
        "furthermore", "moreover", "crucial", "pivotal", "underscore",
        "underscores", "comprehensive", "intricate", "delve", "landscape",
        "highlight", "highlights", "notably", "importantly", "overall",
        "findings suggest", "shed light", "pave the way",
    )

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        feats = []
        for text in X:
            n_chars = max(len(text), 1)
            words = re.findall(r"\b\w+\b", text)
            n_words = max(len(words), 1)
            words_lower = [w.lower() for w in words]

            # Découpage en phrases et mesure de régularité (burstiness)
            sents = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
            sent_lens = [len(re.findall(r"\b\w+\b", s)) for s in sents] or [0]
            mean_sent_len = float(np.mean(sent_lens))
            std_sent_len = float(np.std(sent_lens))
            burstiness = std_sent_len / (mean_sent_len + 1e-5)

            # Diversité lexicale et longueur des mots
            ttr = len(set(words_lower)) / n_words
            mean_word_len = float(np.mean([len(w) for w in words])) if words else 0.0

            # Ponctuation, chiffres précis et majuscules/acronymes
            commas = text.count(",") / n_words
            semicolons = (text.count(";") + text.count(":")) / n_words
            parens = (text.count("(") + text.count(")")) / n_words
            hyphens = text.count("-") / n_words
            digits = len(re.findall(r"\d+(?:\.\d+)?", text)) / n_words
            uppers = sum(1 for c in text if c.isupper()) / n_chars

            # Tournures récurrentes des LLM
            text_low = text.lower()
            ai_cues = sum(text_low.count(m) for m in self.MARQUEURS_IA) / n_words

            feats.append([
                n_words,
                len(sents),
                mean_sent_len,
                std_sent_len,
                burstiness,
                ttr,
                mean_word_len,
                commas,
                semicolons,
                parens,
                hyphens,
                digits,
                uppers,
                ai_cues,
            ])
        return np.asarray(feats, dtype=np.float32)

## 3. Préparation, dédoublonnage et découpage stratifié
Identification automatique de la colonne cible (`label` ou `HumanAI`), suppression des doublons exacts d'abstracts pour éviter toute fuite d'information, et séparation stratifiée (80 % entraînement / 20 % validation).

In [3]:
col_map = {c.lower(): c for c in df_train.columns}
col_abstract = col_map["abstract"]
col_target = col_map.get("label", col_map.get("humanai"))

df_train = df_train.dropna(subset=[col_abstract, col_target]).reset_index(drop=True)
df_train["clean_abstract"] = df_train[col_abstract].apply(nettoyer_html_leger)
df_train["target"] = df_train[col_target].astype(int)

# Dédoublonnage pour éviter la fuite entre train et validation
df_train = df_train.drop_duplicates(subset=["clean_abstract"]).reset_index(drop=True)

print("Répartition des classes (0=Humain, 1=IA) :")
print(df_train["target"].value_counts(normalize=True).round(4))

X_train, X_val, y_train, y_val = train_test_split(
    df_train["clean_abstract"].tolist(),
    df_train["target"].values,
    test_size=0.2,
    random_state=42,
    stratify=df_train["target"].values,
)

Répartition des classes (0=Humain, 1=IA) :
target
0    0.7493
1    0.2507
Name: proportion, dtype: float64


## 4. Entraînement du pipeline hybride et optimisation du seuil F1
Combinaison de trois représentations complémentaires :
* **Stylométrie normalisée** (`StandardScaler`) ;
* **TF-IDF de mots** (unigrammes et bigrammes, sensibles aux tournures) ;
* **TF-IDF de caractères** (3 à 5 caractères, sensibles à la morphologie et à la ponctuation).

Une régression logistique régularisée (`C=2.0`, `class_weight="balanced"`) est ensuite entraînée et son seuil de décision est calibré sur la validation pour maximiser le **F1-score**.

In [4]:
features = FeatureUnion([
    (
        "stylometry",
        Pipeline([
            ("extract", StylometricExtractor()),
            ("scale", StandardScaler()),
        ]),
    ),
    (
        "word_tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=False,
        ),
    ),
    (
        "char_tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=5,
            max_features=60000,
            sublinear_tf=True,
        ),
    ),
])

clf = Pipeline([
    ("features", features),
    (
        "model",
        LogisticRegression(
            C=2.0,
            class_weight="balanced",
            max_iter=1000,
            random_state=42,
        ),
    ),
])

clf.fit(X_train, y_train)
val_probs = clf.predict_proba(X_val)[:, 1]

# Recherche du seuil optimal maximisant le F1-score
best_thresh, best_f1, best_prec = 0.5, -1.0, 0.0
for thresh in np.arange(0.25, 0.80, 0.01):
    preds = (val_probs >= thresh).astype(int)
    f1 = f1_score(y_val, preds)
    prec = precision_score(y_val, preds, zero_division=0)
    if (f1 > best_f1) or (np.isclose(f1, best_f1) and prec > best_prec):
        best_thresh, best_f1, best_prec = float(thresh), float(f1), float(prec)

val_preds = (val_probs >= best_thresh).astype(int)
print(
    f"Validation ML | Seuil optimal : {best_thresh:.2f} | "
    f"F1-score : {best_f1:.4f} | Précision : {best_prec:.4f}"
)
print(classification_report(y_val, val_preds, target_names=["Humain (0)", "IA (1)"]))

Validation ML | Seuil optimal : 0.49 | F1-score : 0.7766 | Précision : 0.7990
              precision    recall  f1-score   support

  Humain (0)       0.92      0.94      0.93      1257
      IA (1)       0.80      0.76      0.78       421

    accuracy                           0.89      1678
   macro avg       0.86      0.85      0.85      1678
weighted avg       0.89      0.89      0.89      1678



## 5. Ré-entraînement sur 100 % des données et export des prédictions
Ré-entraînement du pipeline sur l'intégralité du jeu d'entraînement disponible, application du seuil optimal calibré sur le jeu de test et génération du fichier `Projet2_Script1_ML.csv` (`ID`, `pred`).

In [5]:
if TEST_FILE:
    clf.fit(df_train["clean_abstract"].tolist(), df_train["target"].values)

    df_test = lire_csv_robuste(TEST_FILE)
    test_col_map = {c.lower(): c for c in df_test.columns}
    col_id_test = test_col_map.get("id", df_test.columns[0])
    col_abs_test = test_col_map["abstract"]

    test_texts = df_test[col_abs_test].fillna("").apply(nettoyer_html_leger).tolist()
    test_probs = clf.predict_proba(test_texts)[:, 1]
    test_preds = (test_probs >= best_thresh).astype(int)

    df_sub = pd.DataFrame({
        "ID": df_test[col_id_test],
        "pred": test_preds,
    })
    df_sub.to_csv("Projet2_Script1_ML.csv", index=False)
    print(f"Fichier généré : Projet2_Script1_ML.csv ({df_sub.shape})")
    print("Distribution prédite :", df_sub["pred"].value_counts().to_dict())
else:
    print("Fichier de test introuvable dans :", os.listdir("."))

Fichier généré : Projet2_Script1_ML.csv ((2898, 2))
Distribution prédite : {0: 2035, 1: 863}
